In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
from pyspark.sql import functions as F
# -----------------------------
# 1️⃣ Read CSV file from volume and load into table
# -----------------------------
csv_path = "/Volumes/workspace/demo/mmm_data/sample_data.csv.csv"

df_csv = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(csv_path)

display()
df = df_csv.select(
    [F.col(col).alias(col.upper()) for col in df_csv.columns]
)

display(df)

spark.sql("CREATE DATABASE IF NOT EXISTS demo")

df.write.format("delta").option("mergeSchema", "true").mode("overwrite").saveAsTable("demo.retail_media")

print("✅ Sample data loaded into `demo.retail_media` successfully!")

In [0]:
# Notebook cell 1 — install runtime libs (run on cluster)
%pip install langchain-core databricks-langchain langgraph-supervisor mlflow plotly




In [0]:
%pip install sentence-transformers seaborn matplotlib

In [0]:
dbutils.library.restartPython()

In [0]:
"""
Unified agentic NL→SQL→Spark→Pandas→EDA→Seaborn→Embeddings pipeline
Merged version combining:
 - agentic_with_seaborn_full.py
 - Display/EDA/graph notebook code
"""

from typing import List, Optional, Dict, Any, Tuple
import re
import json
import warnings
from uuid import uuid4

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

# -----------------------------------------------------------------
# Optional embedding library
# -----------------------------------------------------------------
try:
    from sentence_transformers import SentenceTransformer
    _SENTENCE_TRANSFORMER_AVAILABLE = True
except Exception:
    _SENTENCE_TRANSFORMER_AVAILABLE = False

# -----------------------------------------------------------------
# Databricks LLM client (best effort)
# -----------------------------------------------------------------
try:
    from databricks_langchain import ChatDatabricks, DatabricksFunctionClient, set_uc_function_client
    _DB_AVAILABLE = True
except Exception:
    ChatDatabricks = None
    DatabricksFunctionClient = None
    set_uc_function_client = None
    _DB_AVAILABLE = False

# Initialize LLM
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"
_llm = None

if _DB_AVAILABLE:
    try:
        client = DatabricksFunctionClient()
        set_uc_function_client(client)
        _llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)
    except Exception:
        _llm = None

# Table to query
TABLE_NAME = "demo.retail_media"
MAX_SQL_ROWS = 2000
SAFE_ROW_LIMIT = 1000

# -----------------------------------------------------------------
# LLM call wrapper (many fallback interfaces)
# -----------------------------------------------------------------
def call_llm_return_text(prompt: str) -> str:
    if _llm is None:
        raise RuntimeError("LLM not available.")

    # Method 1: generate()
    try:
        if hasattr(_llm, "generate"):
            out = _llm.generate([{"role": "user", "content": prompt}])
            if hasattr(out, "generations"):
                gens = out.generations
                if isinstance(gens, (list, tuple)) and len(gens) > 0:
                    g0 = gens[0]
                    if isinstance(g0, (list, tuple)):
                        return getattr(g0[0], "text", str(g0[0]))
                    else:
                        return getattr(g0, "text", str(g0))
            return str(out)
    except Exception:
        pass

    # Method 2: callable
    try:
        if callable(_llm):
            out = _llm(prompt)
            if isinstance(out, str):
                return out
            if hasattr(out, "content"):
                return out.content
            if hasattr(out, "text"):
                return out.text
            return str(out)
    except Exception:
        pass

    # Method 3: invoke, run, predict...
    for fn in ("invoke", "run", "predict", "chat", "generate_text", "complete"):
        if hasattr(_llm, fn):
            try:
                out = getattr(_llm, fn)(prompt)
                if isinstance(out, str):
                    return out
                if hasattr(out, "content"):
                    return out.content
                if hasattr(out, "text"):
                    return out.text
                return str(out)
            except Exception:
                continue

    raise RuntimeError("LLM invocation failed.")


In [0]:
# -----------------------------
# PART 2/5 — SQL generation, safe runner, date inference, aggregation, plotting
# -----------------------------

# SQL prompt template and generator
SQL_PROMPT_TEXT = (
    "You are an expert SQL generator for Databricks Delta tables.\n\n"
    f"Available table: `{TABLE_NAME}` with columns: DATE, ZONE, REGION, COUNTRY, RETAIL_CHANNEL, RETAILER_NAME, MANUFACTURER, PRODUCT_FAMILY, SPECIES, BRAND, SUB_BRAND, SKU_NAME, MARKETING_CHANNEL, CAMPAIGN_NAME, METRIC, VALUE\n\n"
    "Rules:\n"
    "- Return a single VALID Databricks SQL SELECT statement, no explanation, no markdown fences.\n"
    "- Use uppercase for SQL keywords (SELECT, FROM, WHERE, GROUP BY, ORDER BY).\n"
    "- Use ISO date format 'YYYY-MM-DD' when filtering by date.\n"
    "- If ambiguous, produce a conservative aggregation (SUM(VALUE) with GROUP BY).\n"
    "- Always include an ORDER BY.\n"
    "- Do NOT include destructive statements.\n\n"
    "User question:\n{question}"
)


def sql_agent_call(user_question: str) -> str:
    prompt = SQL_PROMPT_TEXT.format(question=user_question)
    out = call_llm_return_text(prompt)
    up = out.upper()
    if "SELECT" in up:
        idx = up.find("SELECT")
        cand = out[idx:]
        cand = re.sub(r"```(?:sql)?", "", cand, flags=re.IGNORECASE).strip()
        cand = re.sub(r"```$", "", cand).strip()
        return cand
    return out.strip()


# Safe SQL runner using Spark
def run_sql(query: str, max_rows: int = MAX_SQL_ROWS) -> Dict[str, Any]:
    if not isinstance(query, str):
        raise ValueError("query must be a string")
    q_upper = query.upper()
    forbidden = ["DROP ", "DELETE ", "TRUNCATE ", "ALTER ", "SHUTDOWN", "GRANT ", "REVOKE ", "CREATE TABLE", "CREATE DATABASE"]
    for kw in forbidden:
        if kw in q_upper:
            raise ValueError(f"Refusing to run query containing forbidden keyword: {kw.strip()}")
    if "SELECT" not in q_upper:
        raise ValueError("Only SELECT queries are allowed.")
    try:
        df = spark.sql(query)
    except Exception as e:
        raise RuntimeError(f"spark.sql failed: {e}\nQuery was: {query}")
    try:
        pdf = df.limit(int(max_rows)).toPandas()
    except Exception:
        pdf = df.toPandas()
    return {"query": query, "rows": json.loads(pdf.to_json(orient="records", date_format="iso")), "rowcount": len(pdf), "columns": list(pdf.columns)}


# -----------------------------
# Date handling / inference utilities
# -----------------------------

def safe_datetime_infer(df: pd.DataFrame, prefer_cols: Optional[List[str]] = None) -> Tuple[pd.DataFrame, Optional[str]]:
    df = df.copy()
    cols_lower = {c.lower(): c for c in df.columns}

    # Preferred columns first (case-insensitive match)
    if prefer_cols:
        for pc in prefer_cols:
            if pc.lower() in cols_lower:
                col_name = cols_lower[pc.lower()]
                df[col_name] = pd.to_datetime(df[col_name], errors="coerce", infer_datetime_format=True)
                return df, col_name

    # Year + month combination (common aggregation pattern)
    if 'year' in cols_lower and 'month' in cols_lower:
        ycol = cols_lower['year']
        mcol = cols_lower['month']
        def combine_ym(y, m):
            try:
                if pd.isna(y) or pd.isna(m):
                    return pd.NaT
                y = int(y)
                if isinstance(m, str):
                    mm = m.strip()
                    if mm.isdigit():
                        m_int = int(mm)
                    else:
                        try:
                            m_int = pd.to_datetime(mm, format="%b").month
                        except Exception:
                            try:
                                m_int = pd.to_datetime(mm, format="%B").month
                            except Exception:
                                return pd.NaT
                else:
                    m_int = int(m)
                return pd.Timestamp(year=y, month=m_int, day=1)
            except Exception:
                return pd.NaT
        df['date'] = [combine_ym(yy, mm) for yy, mm in zip(df[ycol].values, df[mcol].values)]
        return df, 'date'

    # Try to find any column with keywords
    candidates = [c for k, c in cols_lower.items() if any(x in k for x in ('date', 'dt', 'timestamp', 'time', 'day'))]
    candidates = sorted(candidates, key=lambda c: (0 if c.lower() == 'date' else 1, c))
    for c in candidates:
        df[c] = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)
        if df[c].notna().sum() > 0:
            return df, c

    # Heuristic parse for patterns like YYYYMM, YYYY-MM
    pattern1 = re.compile(r'^\d{4}[-/]?\d{2}([-/]?\d{2})?$')
    for c in df.columns:
        sample = df[c].dropna().astype(str).head(10).tolist()
        if any(pattern1.match(s) for s in sample):
            try:
                df[c] = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)
                if df[c].notna().sum() > 0:
                    return df, c
            except Exception:
                pass

    return df, None


def infer_time_freq_from_series(series: pd.Series) -> str:
    s = series.dropna().sort_values()
    if len(s) < 2:
        return 'D'
    diffs = (s.iloc[1:] - s.iloc[:-1]).map(lambda x: x.days if pd.notna(x) else np.nan).dropna()
    if diffs.empty:
        return 'D'
    median = diffs.median()
    if median <= 1:
        return 'D'
    if median <= 7:
        return 'W'
    if median <= 31:
        return 'M'
    if median <= 120:
        return 'Q'
    return 'Y'


# -----------------------------
# Aggregation utilities
# -----------------------------

def aggregate_time_series(df: pd.DataFrame, x_col: str, y_cols: List[str], freq: str = 'M', agg: str = 'sum') -> pd.DataFrame:
    df = df.copy()
    if x_col not in df.columns:
        raise ValueError(f"x_col {x_col} not in df")
    df[x_col] = pd.to_datetime(df[x_col], errors='coerce')
    df = df.dropna(subset=[x_col])
    df = df.set_index(x_col)
    if agg == 'sum':
        agg_df = df[y_cols].resample(freq).sum(min_count=1)
    elif agg == 'mean':
        agg_df = df[y_cols].resample(freq).mean()
    else:
        agg_df = df[y_cols].resample(freq).agg(agg)
    agg_df = agg_df.reset_index()
    return agg_df


# -----------------------------
# Plotting with Seaborn (time series)
# -----------------------------

def plot_seaborn_time_series(
    df: pd.DataFrame,
    x_col: str,
    y_cols: List[str],
    agg: str = 'sum',
    freq: Optional[str] = None,
    figsize: Tuple[int, int] = (14, 6),
    annotate: bool = False,
    palette: Optional[List[str]] = None,
    title: Optional[str] = None,
    xlabel: Optional[str] = None,
    ylabel: Optional[str] = None,
    rotate_xticks: int = 45
):
    df = df.copy()
    if x_col not in df.columns:
        raise ValueError(f"x_col `{x_col}` not in dataframe")
    df[x_col] = pd.to_datetime(df[x_col], errors='coerce')
    if df[x_col].isna().all():
        raise ValueError(f"All values in {x_col} are NaT after coercion")

    if freq is None:
        freq = infer_time_freq_from_series(df[x_col])
        map_to_alias = {'D': 'D', 'W': 'W', 'M': 'M', 'Q': 'Q', 'Y': 'Y'}
        freq = map_to_alias.get(freq, 'M')

    agg_df = aggregate_time_series(df, x_col, y_cols, freq=freq, agg=agg)

    fig, ax = plt.subplots(figsize=figsize)
    if palette is None:
        palette = sns.color_palette("tab10", n_colors=max(3, len(y_cols)))

    xcol_plot = agg_df.columns[0]

    for i, yc in enumerate(y_cols):
        if yc not in agg_df.columns:
            warnings.warn(f"{yc} not found in aggregated DF; skipping")
            continue
        sns.lineplot(data=agg_df, x=xcol_plot, y=yc, ax=ax, label=yc, marker='o', linewidth=2, color=palette[i % len(palette)])

    if title:
        ax.set_title(title)
    ax.set_xlabel(xlabel or x_col)
    ax.set_ylabel(ylabel or (", ".join(y_cols)))
    plt.xticks(rotation=rotate_xticks)
    plt.tight_layout()

    if annotate:
        for i, yc in enumerate(y_cols):
            if yc not in agg_df.columns:
                continue
            for x, y in zip(agg_df[xcol_plot], agg_df[yc]):
                if pd.isna(y):
                    continue
                ax.text(x, y, f"{y:.2f}", fontsize=8, ha='center', va='bottom', rotation=0)

    # Reduce tick crowding
    xticks = ax.get_xticks()
    n_ticks = 10
    if len(xticks) > n_ticks:
        sel = np.linspace(0, len(xticks) - 1, n_ticks, dtype=int)
        ax.set_xticks([xticks[i] for i in sel])

    plt.legend(title="Series")
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.show()
    return fig, agg_df


In [0]:
# -----------------------------
# PART 3/5 — EDA, Embeddings, High-level integration function
# -----------------------------

# -----------------------------
# EDA utilities
# -----------------------------
def perform_eda(df: pd.DataFrame, max_unique_for_value_counts: int = 20) -> Dict[str, Any]:
    eda = {}
    eda['shape'] = df.shape
    eda['dtypes'] = df.dtypes.apply(lambda x: str(x)).to_dict()
    missing = df.isna().sum()
    eda['missing_count'] = missing.to_dict()
    eda['missing_percent'] = (missing / len(df) * 100).round(2).to_dict()
    eda['basic_numeric'] = df.select_dtypes(include=[np.number]).describe().to_dict()

    cat_top = {}
    for c in df.select_dtypes(include=['object', 'category']).columns:
        vc = df[c].value_counts(dropna=False)
        if vc.size <= max_unique_for_value_counts:
            cat_top[c] = vc.to_dict()
        else:
            cat_top[c] = {str(v): int(cnt) for v, cnt in vc.head(10).items()}
    eda['categorical_top_values'] = cat_top

    numeric = df.select_dtypes(include=[np.number])
    if numeric.shape[1] >= 2:
        corr = numeric.corr().abs()
        corr_values = []
        for i in range(len(corr.columns)):
            for j in range(i + 1, len(corr.columns)):
                corr_values.append((corr.index[i], corr.columns[j], corr.iloc[i, j]))
        corr_values = sorted(corr_values, key=lambda x: -x[2])
        eda['top_correlations'] = corr_values[:20]
    else:
        eda['top_correlations'] = []

    suggestions = []
    if any('date' in c.lower() for c in df.columns):
        suggestions.append("Time series plots: aggregate numeric metrics by month/quarter/year.")
    if len(df.select_dtypes(include=['object', 'category']).columns) > 0:
        suggestions.append("Bar charts for categorical distributions (value_counts).")
    if len(numeric.columns) > 0:
        suggestions.append("Histograms and boxplots to examine numeric distributions and outliers.")
    eda['suggestions'] = suggestions

    # Print short human-friendly summary
    print("EDA Summary")
    print("----------")
    print("Shape:", eda['shape'])
    non_zero_missing = {k: v for k, v in eda['missing_percent'].items() if v > 0}
    print("Missing % (non-zero):", dict(list(non_zero_missing.items())[:10]))
    print("Numeric columns:", list(numeric.columns))
    print("Categorical columns:", list(cat_top.keys()))
    if eda['top_correlations']:
        print("Top correlation pairs:", eda['top_correlations'][:3])
    print("Suggestions:", suggestions)

    return eda


# -----------------------------
# Embedding utilities
# -----------------------------
def generate_text_embeddings(df: pd.DataFrame, text_column: str, model_name: str = 'all-MiniLM-L6-v2', output_col_prefix: str = 'embedding') -> pd.DataFrame:
    if not _SENTENCE_TRANSFORMER_AVAILABLE:
        raise RuntimeError("sentence-transformers not available. Install with: %pip install sentence-transformers")
    if text_column not in df.columns:
        raise ValueError(f"text_column {text_column} not in DataFrame")
    model = SentenceTransformer(model_name)
    texts = df[text_column].fillna("").astype(str).tolist()
    batch_size = 128
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        emb = model.encode(batch, show_progress_bar=False, convert_to_numpy=True)
        embeddings.extend(list(emb))
    out_col = f"{output_col_prefix}_{model_name}"
    df[out_col] = embeddings
    return df


# -----------------------------
# High-level integration function
# -----------------------------
def nl_to_chart_agentic_with_seaborn(
    user_question: str,
    sql_override: Optional[str] = None,
    prefer_date_cols: Optional[List[str]] = None,
    x_col: Optional[str] = None,
    y_cols: Optional[List[str]] = None,
    freq: Optional[str] = None,
    agg: str = 'sum',
    do_eda: bool = True,
    do_embeddings: bool = False,
    text_column_for_embeddings: Optional[str] = None,
    embed_model_name: str = 'all-MiniLM-L6-v2',
    max_rows: int = MAX_SQL_ROWS,
) -> Dict[str, Any]:
    """
    End-to-end function:
    - Generates SQL from user_question (unless sql_override provided)
    - Executes SQL and returns DataFrame
    - Performs EDA (optional)
    - Optionally generates embeddings
    - Infers date column and plots Seaborn time-series or categorical chart(s)

    Returns a dictionary with keys: sql, df (pandas), df_agg (if time series), eda (if run), fig (matplotlib fig), trace
    """
    trace: Dict[str, Any] = {"user_question": user_question}
    # 1) SQL generation
    generated_sql = None
    if sql_override:
        generated_sql = sql_override
        trace['sql_source'] = 'override'
    else:
        try:
            generated_sql = sql_agent_call(user_question)
            trace['sql_source'] = 'llm'
        except Exception as e:
            raise RuntimeError("LLM SQL generation failed and no sql_override provided: " + str(e))

    trace['generated_sql'] = generated_sql

    # 2) Execute SQL
    sql_result = run_sql(generated_sql, max_rows=max_rows)
    rows = sql_result.get('rows', [])
    df = pd.DataFrame(rows)
    trace['rowcount'] = sql_result.get('rowcount', 0)

    if df.empty:
        # Return early with structure but no data
        return {"sql": generated_sql, "df": df, "df_agg": None, "eda": None, "fig": None, "trace": trace}

    # 3) EDA
    eda = None
    if do_eda:
        eda = perform_eda(df)
        trace['eda'] = 'done'

    # 4) Embeddings
    if do_embeddings:
        if not text_column_for_embeddings:
            raise ValueError("text_column_for_embeddings must be provided if do_embeddings=True")
        df = generate_text_embeddings(df, text_column_for_embeddings, model_name=embed_model_name)
        trace['embeddings_col'] = f"embedding_{embed_model_name}"

    # 5) Infer date column
    df2, detected_date_col = safe_datetime_infer(df, prefer_cols=prefer_date_cols)
    trace['detected_date_col'] = detected_date_col

    # 6) Determine x and y
    chosen_x = x_col or detected_date_col
    if not chosen_x:
        non_numeric = [c for c in df2.columns if not pd.api.types.is_numeric_dtype(df2[c])]
        chosen_x = non_numeric[0] if non_numeric else df2.columns[0]
    trace['chosen_x'] = chosen_x

    if y_cols:
        chosen_y = [c for c in y_cols if c in df2.columns]
    else:
        numeric_cols = [c for c in df2.columns if pd.api.types.is_numeric_dtype(df2[c])]
        chosen_y = numeric_cols[:3]
    trace['chosen_y'] = chosen_y

    fig = None
    df_agg = None
    try:
        if chosen_x and chosen_y and pd.api.types.is_datetime64_any_dtype(df2[chosen_x]):
            fig, df_agg = plot_seaborn_time_series(df2, x_col=chosen_x, y_cols=chosen_y, agg=agg, freq=freq, title=user_question)
        elif chosen_x and chosen_y:
            fig, ax = plt.subplots(figsize=(12, 6))
            melted = df2.melt(id_vars=[chosen_x], value_vars=chosen_y, var_name='metric', value_name='value')
            sns.barplot(data=melted, x=chosen_x, y='value', hue='metric', ax=ax)
            ax.set_title(user_question)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
        else:
            pass
    except Exception as e:
        trace['plot_error'] = str(e)

    return {"sql": generated_sql, "df": df2, "df_agg": df_agg, "eda": eda, "fig": fig, "trace": trace}


In [0]:
# -----------------------------
# PART 4/5 — Notebook Display Helper (SQL + pandas + EDA + Graph)
# -----------------------------
# This block merges your full second snippet into the unified module.
# After running nl_to_chart_agentic_with_seaborn(), call:
#     display_full_output(out)
# to display SQL + pandas + EDA JSON + Seaborn plot.

from IPython.display import display, Image
import io


def display_full_output(out: Dict[str, Any]):
    """
    Unified notebook-display function:
    - Prints SQL
    - Prints EDA JSON
    - Displays pandas DataFrame
    - Ensures a proper datetime column
    - Runs Seaborn time-series plot again safely
    """

    # -------------------------------------------
    # 1) SQL
    # -------------------------------------------
    print("\n=== SQL Used ===")
    print(out.get("sql", ""))

    # -------------------------------------------
    # 2) DataFrame
    # -------------------------------------------
    df = out.get("df")
    if df is None:
        print("No DataFrame returned.")
        return

    print("\n=== DataFrame head ===")
    display(df.head(10))

    # -------------------------------------------
    # 3) EDA (JSON)
    # -------------------------------------------
    eda = out.get("eda")
    print("\n=== Full EDA (JSON) ===")
    try:
        print(json.dumps(eda, indent=2))
    except Exception:
        print("No EDA object or failed to pretty-print EDA.")

    # -------------------------------------------
    # 4) Ensure a datetime column exists
    # -------------------------------------------
    if 'date' not in df.columns:
        # Try YEAR + MONTH (uppercase)
        if 'YEAR' in df.columns and 'MONTH' in df.columns:
            df['date'] = pd.to_datetime(
                df['YEAR'].astype(int).astype(str) + '-' +
                df['MONTH'].astype(int).astype(str).str.zfill(2) + '-01',
                errors='coerce'
            )
            print("Created 'date' column from YEAR + MONTH.")

        # Try lowercase year + month
        elif 'year' in df.columns and 'month' in df.columns:
            df['date'] = pd.to_datetime(
                df['year'].astype(int).astype(str) + '-' +
                df['month'].astype(int).astype(str).str.zfill(2) + '-01',
                errors='coerce'
            )
            print("Created 'date' column from year + month.")

        else:
            # Fallback — try auto detection
            df2, date_col = safe_datetime_infer(df)
            if date_col:
                df['date'] = df2[date_col]
                print(f"Detected date column: {date_col}")
            else:
                print("Warning: No valid date column found. Plotting may fail.")

    # -------------------------------------------
    # 5) Determine available numeric Y columns
    # -------------------------------------------
    # User requested metrics:
    default_targets = ['TOTAL_SPENDS', 'TOTAL_HH_GRPS', 'SPENDS', 'HH_GRPS', 'VALUE']

    y_cols = [c for c in default_targets if c in df.columns]
    if not y_cols:
        # fallback: choose first numeric columns
        y_cols = df.select_dtypes(include=['number']).columns.tolist()[:2]

    print("\n=== Plotting these numeric columns ===")
    print(y_cols)

    # -------------------------------------------
    # 6) Final Seaborn time-series plot
    # -------------------------------------------
    try:
        fig, df_agg = plot_seaborn_time_series(
            df,
            x_col='date',
            y_cols=y_cols,
            freq='M',
            annotate=True,
            title="Time Series Plot"
        )

        # Display the figure as a PNG
        buf = io.BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight')
        buf.seek(0)
        display(Image(data=buf.getvalue()))
        buf.close()

        print("\n=== Aggregated DataFrame (df_agg) ===")
        display(df_agg.head(10))

    except Exception as e:
        print("Plotting failed:", e)


In [0]:
%sql
SELECT 
  DATE_TRUNC('month', DATE) AS MONTH_YEAR, 
  REGION, 
  SUM(VALUE) AS SPENDS
FROM 
  demo.retail_media
WHERE 
  SPECIES = 'CAT' AND METRIC = 'SPENDS'
GROUP BY 
  DATE_TRUNC('month', DATE), 
  REGION
ORDER BY 
  MONTH_YEAR, 
  REGION

In [0]:
# -----------------------------
# PART 5/5 — Script Footer + Example Runner
# -----------------------------
# You can now call:
# 
#     out = nl_to_chart_agentic_with_seaborn("your NL question here")
#     display_full_output(out)
#
# inside any Databricks notebook.


if __name__ == '__main__':
    print("\n============================")
    print(" Running Example Pipeline ")
    print("============================\n")

    example_q = (
        "Show total SPENDS and total HH_GRPS for DOG species. "
        "Plot the values for both metrics in a time series chart month and year wise. "
        "Month/year on x-axis, values on y-axis."
    )

    try:
        out = nl_to_chart_agentic_with_seaborn(
            example_q,
            y_cols=['SPENDS', 'HH_GRPS'],
            freq='M',
            do_eda=True,
            do_embeddings=False
        )

        print("\n>>> SQL Used:\n", out.get("sql"))
        print("\n>>> DataFrame Shape:", out.get("df").shape if out.get("df") is not None else None)

        # Display SQL + pandas + eda + graph
        print("\n\n>>> Displaying Full Output...\n")
        display_full_output(out)

    except Exception as err:
        print("\nERROR running the example pipeline:\n", err)


In [0]:
%sql
SELECT 
  DATE_TRUNC('month', DATE) AS MONTH, 
  CAMPAIGN_NAME, 
  SUM(VALUE) AS SPENDS
FROM 
  demo.retail_media
WHERE 
  SPECIES = 'DOG' 
  AND METRIC = 'SPENDS'
GROUP BY 
  DATE_TRUNC('month', DATE), 
  CAMPAIGN_NAME
ORDER BY 
  MONTH ASC;

In [0]:
%sql
SELECT 
  EXTRACT(YEAR FROM DATE) AS YEAR,
  SUM(CASE WHEN METRIC = 'SPENDS' THEN VALUE ELSE 0 END) AS TOTAL_SPENDS,
  SUM(CASE WHEN METRIC = 'HH_GRPS' THEN VALUE ELSE 0 END) AS TOTAL_HH_GRPS
FROM 
  demo.retail_media
WHERE 
  SPECIES = 'DOG'
GROUP BY 
  EXTRACT(YEAR FROM DATE)
ORDER BY 
  YEAR

In [0]:
%sql
SELECT 
  EXTRACT(YEAR FROM DATE) AS YEAR,
  SUM(CASE WHEN METRIC = 'spends' THEN VALUE ELSE 0 END) AS total_spends,
  SUM(CASE WHEN METRIC = 'hh_grps' THEN VALUE ELSE 0 END) AS total_hh_grps
FROM 
  demo.retail_media
GROUP BY 
  EXTRACT(YEAR FROM DATE)
ORDER BY 
  YEAR

In [0]:
Trace: {
  "supervisor_prompt": "\nYou are a supervisor responsible for coordinating subagents to answer the user's question.\nAvailable subagents:\n- sql-generator-agent: generates a Databricks SQL SELECT statement for table `demo.retail_media`.\n- chart-generator-agent: given a small JSON sample and the user's question, returns executable Plotly Python code (variable 'fig' and fig.show()).\n\nSupervisor instructions:\n1) Read the user's request.\n2) Decide which subagent(s) to call and in what order.\n3) Ask the sql-generator-agent to produce the SQL when needed.\n4) After SQL is executed by the environment, ask the chart-generator-agent to create visualization code using a sample of the results.\n5) Prefer safe, read-only operations.\n6) Return a short orchestration trace (which agent was used) and a final result pointer.\n",
  "actions": [
    {
      "supervisor_decision": "sql-generator-agent",
      "raw": "sql-generator-agent"
    },
    {
      "sql_generated_raw": "SELECT \n  EXTRACT(YEAR FROM DATE) AS YEAR,\n  EXTRACT(MONTH FROM DATE) AS MONTH,\n  SUM(CASE WHEN METRIC = 'SPENDS' THEN VALUE ELSE 0 END) AS TOTAL_SPENDS,\n  SUM(CASE WHEN METRIC = 'HH_GRPS' THEN VALUE ELSE 0 END) AS TOTAL_HH_GRPS\nFROM \n  demo.retail_media\nWHERE \n  SPECIES = 'DOG'\nGROUP BY \n  EXTRACT(YEAR FROM DATE),\n  EXTRACT(MONTH FROM DATE)\nORDER BY \n  YEAR,\n  MONTH"
    },
    {
      "generated_sql": "SELECT \n  EXTRACT(YEAR FROM DATE) AS YEAR,\n  EXTRACT(MONTH FROM DATE) AS MONTH,\n  SUM(CASE WHEN METRIC = 'SPENDS' THEN VALUE ELSE 0 END) AS TOTAL_SPENDS,\n  SUM(CASE WHEN METRIC = 'HH_GRPS' THEN VALUE ELSE 0 END) AS TOTAL_HH_GRPS\nFROM \n  demo.retail_media\nWHERE \n  SPECIES = 'DOG'\nGROUP BY \n  EXTRACT(YEAR FROM DATE),\n  EXTRACT(MONTH FROM DATE)\nORDER BY \n  YEAR,\n  MONTH"
    },
    {
      "executed_rowcount": 5
    },
    {
      "chart_choice": "LINE"
    },
    {
      "chart_code_snippet": "import pandas as pd\nimport plotly.express as px\n\n# Ensure date columns are datetime\ndf['DATE'] = pd.to_datetime(df[['YEAR', 'MONTH']].assign(DAY=1))\n\n# Create a line chart\nfig = px.line(df, x='DATE', y=['TOTAL_SPENDS', 'TOTAL_HH_GRPS'])\n\n# Update layout\nfig.update_layout(\n    title='Time Series Char"
    },
    {
      "chart_exec_error": "__import__ not found"
    }
  ]
}
Rows: (5, 4)


In [0]:
# 2) create supervisor and wrapper
EXAMPLE_LLM_ENDPOINT = "databricks-claude-3-7-sonnet"  # replace if needed
supervisor = None
agent_wrapper = None
try:
    supervisor, agent_wrapper = create_and_register_supervisor(
        EXAMPLE_LLM_ENDPOINT,
        register_mlflow=False
    )
except RuntimeError as e:
    print(f"Error: {e}")

# 4) stream the supervisor — this yields update events from the compiled graph
if supervisor is not None:
    print("Streaming supervisor output:\n")
    for _, events in supervisor.stream(
        {"messages": cc_msgs},
        stream_mode=["updates"]
    ):
        for node_name, payload in events.items():
            for msg in payload.get("messages", []):
                print(f"[{node_name}] -> {getattr(msg, 'content', msg)}")
        time.sleep(0.1)
else:
    print("Supervisor was not created. Please check the error above.")

In [0]:
# Install the likely-needed packages for LangGraph supervisor + Databricks # 1) install needed libraries (notebook-scoped). Prefer cluster install via UI for stability.
%pip install --upgrade pip
%pip install databricks-langchain langgraph langgraph-supervisor langchain-core mlflow



In [0]:
dbutils.library.restartPython()

In [0]:
import importlib, agent as aup
importlib.reload(aup)

# pass your running model endpoint name (or any endpoint that ChatDatabricks accepts)
example_llm_endpoint = "databricks-claude-3-7-sonnet"

# create the supervisor and wrapper. register_mlflow=False to avoid model registration if you don't want it.
supervisor, wrapper = aup.create_and_register_supervisor(example_llm_endpoint, register_mlflow=False)

print("Supervisor:", type(supervisor))
print("Wrapper:", type(wrapper))



In [0]:
from mlflow.types.responses import ResponsesAgentRequest

input_example = {
    "input": [
        {"role": "user", "content": "what tools do you have access to"}
    ]
}

req = ResponsesAgentRequest(**input_example)
resp = wrapper.predict(req)
print("Response:", resp)

In [0]:
from mlflow.types.responses import ResponsesAgentRequest

req = ResponsesAgentRequest(
    input=[{"role": "user", "content": "Summarize monthly spends for India"}]
)

# wrapper is the ResponsesAgent-like returned from create_and_register_supervisor
resp = wrapper.predict(req)
print(resp)

In [0]:
# 1. Write agent.py to a local file the driver can read
agent_source = "genai/agent.py"
local_agent_path = "/tmp/agent.py"   # use /tmp on driver
with open(local_agent_path, "w") as f:
    f.write(agent_source)
print("Wrote agent to", local_agent_path)


In [0]:
import mlflow, importlib
from mlflow import pyfunc
from mlflow.models.signature import ModelSignature
from mlflow.types import Schema, ColSpec

# CONFIG
WRAPPER_PATH = "/tmp/agent_wrapper_for_uc.py"
AGENT_PY_PATH = "agent.py"   # change if your agent file is elsewhere
REGISTERED_MODEL_NAME = "workspace.default.agent_model_registry_name"  # or UC target if using UC

# minimal signature: one string input and string JSON output (Unity Catalog likes signatures)
signature = ModelSignature(inputs=Schema([ColSpec("string","user_question")]), outputs=Schema([ColSpec("string","response_json")]))

conda_env = {
    'name': 'mlflow-env',
    'channels': ['defaults', 'conda-forge'],
    'dependencies': [
        'python=3.12.3',   # ensure python version 3.10 here
        'pip',
        {
            'pip': [
                'mlflow',
                'pandas',
                'numpy',
                'seaborn',
                'matplotlib',
                'cloudpickle==3.0.0',   # pin cloudpickle to serving runtime version
                # add any other pips your agent needs
            ]
        },
    ],
}

# load wrapper class module so mlflow sees source
import importlib.util, sys
spec = importlib.util.spec_from_file_location("agent_wrapper_for_uc", WRAPPER_PATH)
wrapper_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(wrapper_mod)
AgentPyFunc = wrapper_mod.AgentPyFunc

with mlflow.start_run() as run:
    model_info = pyfunc.log_model(
        artifact_path="agent_pyfunc",
        python_model=AgentPyFunc(),
        conda_env=conda_env,
        code_paths=[WRAPPER_PATH, AGENT_PY_PATH],
        registered_model_name=REGISTERED_MODEL_NAME,
        signature=signature,
        input_example={"user_question": "Show monthly VALUE for India last 6 months"}
    )
    print("Logged model URI:", model_info.model_uri)
    print("Run id:", run.info.run_id)


In [0]:
# ------------- CONFIG - change these -------------
# Fill in the UC catalog, schema and desired model name
catalog = "workspace"      # e.g. "main" or "hive_metastore" or your org's UC catalog
schema = "default"           # e.g. "analytics"
model_name = "agent_model_registry_name"       # final model name in UC (no spaces)
# If you already have a model_uri from a previous mlflow.log_model call, put it here.
# Example: model_uri = model_info.model_uri  OR "runs:/<run_id>/agent_pyfunc"
model_uri = model_info.model_uri   # e.g. "runs:/<run_id>/agent_pyfunc"
# -------------------------------------------------

from time import sleep
import mlflow
from mlflow.exceptions import MlflowException
from mlflow.tracking import MlflowClient

# Ensure MLflow uses Unity Catalog as registry
mlflow.set_registry_uri("databricks-uc")

# Compose the UC model name in the required format: <catalog>.<schema>.<model>
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"
print("Unity Catalog model name will be:", UC_MODEL_NAME)

# Sanity checks
if not model_uri or model_uri.startswith("<"):
    raise ValueError("Please set model_uri to the artifact you logged (e.g. model_info.model_uri or 'runs:/<run_id>/agent_pyfunc').")

client = MlflowClient()

# Register the model - this creates a new registered model in UC and a new model version
try:
    print("Registering model:", model_uri, "->", UC_MODEL_NAME)
    mv = mlflow.register_model(model_uri=model_uri, name=UC_MODEL_NAME)
    # mlflow.register_model returns a ModelVersion (object with .version). Use MlflowClient to inspect it.
    print("Model registration requested. Version id (server may create async):", mv.version)
except MlflowException as e:
    print("Register call failed with MlflowException:", e)
    raise

# Wait for model version to become available (simple poll)
model_version = mv.version
for i in range(60):   # wait up to ~3-5 minutes (adjust)
    try:
        vinfo = client.get_model_version(name=UC_MODEL_NAME, version=str(model_version))
        status = vinfo.status
        print(f"Polling: attempt {i+1}, status={status}")
        if status in ("READY", "READY_FOR_REVIEW", "READY_TO_DEPLOY", "AVAILABLE"):
            print("Model version is ready:", vinfo)
            break
        elif status == "FAILED_REGISTRATION" or status == "FAILED":
            raise RuntimeError(f"Model version registration failed: {vinfo}")
    except Exception as ex:
        # sometimes the server needs a few seconds to list/create the new version
        print("Waiting for model version to appear...", str(ex))
    sleep(5)
else:
    print("Timed out waiting for model version. Check Model Registry UI for details.")

print("Done. Registered model:", UC_MODEL_NAME, "version:", model_version)
print("You can view it in the Databricks Model Registry / Unity Catalog UI.")


In [0]:
mlflow.set_registry_uri("databricks-uc")

catalog = "workspace"
schema = "default"
model_name = "agent_model_registry_name"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"
model_uri = model_info.model_uri

uc_registered_model_info = mlflow.register_model(
    model_uri=model_uri,
    name=UC_MODEL_NAME
)

In [0]:
import sys, pkgutil, importlib
print("Python:", sys.version)
try:
    import cloudpickle
    print("cloudpickle:", cloudpickle.__version__)
except Exception:
    print("cloudpickle not installed in this runtime")


In [0]:
# Ensure you have: databricks-langchain, langgraph, mlflow, pandas, seaborn, matplotlib installed

from databricks_langchain import ChatDatabricks, DatabricksFunctionClient, set_uc_function_client, GenieAgent, UCFunctionToolkit
from langgraph.graph import StateGraph
from langgraph_supervisor import create_supervisor
from langchain_core.runnables import Runnable
from mlflow.pyfunc import ResponsesAgent

import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt, json, re, io
sns.set_theme(style="whitegrid")

# ----- Databricks clients -----
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"

# ----- Agent definitions -----

class SQLGenieAgent:
    """Agent that generates SQL via a Databricks endpoint/Genie."""
    def __init__(self, endpoint=LLM_ENDPOINT_NAME):
        self.llm = ChatDatabricks(endpoint=endpoint)
        self.prompt_template = """You are an expert SQL generator ...""" # Use your original template

    def predict(self, request):
        question = request['input'] if isinstance(request['input'], str) else request['input'][0]
        prompt = self.prompt_template.format(question=question)
        sql = self.llm.generate([{"role":"user","content":prompt}]).generations[0][0].text
        return {"sql": sql}

class EDAAgent:
    """Agent for automated exploratory analysis."""
    def predict(self, request):
        df = request['df']
        return {
            "shape": df.shape,
            "dtypes": df.dtypes.apply(str).to_dict(),
            "missing": df.isnull().sum().to_dict(),
            "describe": df.describe(include='all').to_dict()
        }

class PlotAgent:
    """Agent that produces Seaborn plots as requested."""
    def predict(self, request):
        df, x_col, y_cols = request['df'], request['x_col'], request['y_cols']
        fig, ax = plt.subplots(figsize=(14, 6))
        for yc in y_cols:
            sns.lineplot(data=df, x=x_col, y=yc, ax=ax, label=yc)
        plt.legend()
        buf = io.BytesIO()
        fig.savefig(buf, format="png")
        buf.seek(0)
        return {"plot": buf.getvalue()}  # or just return "plot rendered" if running in notebook

class EmbeddingAgent:
    """Agent for producing sentence-transformer embeddings."""
    def predict(self, request):
        col, model = request['text_column'], request.get('model', 'all-MiniLM-L6-v2')
        texts = request['df'][col].fillna("").astype(str).tolist()
        from sentence_transformers import SentenceTransformer
        embs = SentenceTransformer(model).encode(texts)
        request['df'][f"embedding_{model}"] = embs
        return {"embeddings": embs}

# ----- Agent registry -----
AGENTS = {
    "sql_genie": SQLGenieAgent(),
    "eda": EDAAgent(),
    "plot": PlotAgent(),
    "embedding": EmbeddingAgent(),
    # add more agents as needed
}

# ----- LangGraph Supervisor -----
def agentic_supervisor(request, agents=AGENTS):
    """
    Supervisor function: takes a request dict and dispatches to correct agent.
    Agent selection logic: match keywords, then call agent's predict().
    """
    question = request.get("question", "").lower()
    if "sql" in question or "query" in question:
        agent = agents["sql_genie"]
    elif "eda" in question or "analysis" in question or "summary" in question:
        agent = agents["eda"]
    elif "plot" in question or "trend" in question or "chart" in question:
        agent = agents["plot"]
    elif "embedding" in question or "vector" in question:
        agent = agents["embedding"]
    else:
        agent = None
    if agent:
        return agent.predict(request)
    else:
        return {"error":"No suitable agent found"}

# ----- LangGraph as pipeline workflow -----
# Example of wiring multiple agents as StateGraph, if you wish complex flows
def build_agent_graph():
    graph = StateGraph()
    graph.add_node("SQLGenie", AGENTS["sql_genie"].predict)
    graph.add_node("EDA", AGENTS["eda"].predict)
    graph.add_node("Plot", AGENTS["plot"].predict)
    graph.add_node("Embedding", AGENTS["embedding"].predict)
    # add edges as needed; e.g., output of SQLGenie feeds into EDA, Plot, etc.
    return graph

# ----- Genie agent as part of graph -----
def genie_space_agent(space_id):
    return GenieAgent(genie_space_id=space_id, genie_agent_name="GenieSpace")

# ----- Unified API -----
def run_multiagent_pipeline(
    question,
    dataframe=None,
    plot_columns=None
):
    request = {"question": question}
    if dataframe is not None:
        # Convert Spark DataFrame to Pandas DataFrame if needed
        if hasattr(dataframe, "toPandas"):
            dataframe = dataframe.toPandas()
        # Standardize column names to lowercase
        dataframe.columns = [col.lower() for col in dataframe.columns]
        request["df"] = dataframe
    if plot_columns is not None:
        # Also standardize plot_columns to lowercase
        plot_columns = [col.lower() for col in plot_columns]
        request["x_col"] = plot_columns[0]
        request["y_cols"] = plot_columns[1:]
    result = agentic_supervisor(request)
    return result

# ----- Example usage -----
if __name__ == "__main__":
    # Example dataframe
    df = spark.sql("SELECT * FROM demo.retail_media")
    pdf = df.toPandas()
    print(run_multiagent_pipeline("Please show EDA analysis", dataframe=df))
    print(run_multiagent_pipeline("plot the data for greenies and temptations", dataframe=df, plot_columns=["date","value"]))

